In [16]:
# !pip install --no-cache-dir pandas pyarrow
# !pip install openpyxl
!pip install tqdm



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [17]:
import pandas as pd
import requests
import json
import time
from tqdm import tqdm

In [2]:
gold_company = pd.read_excel("/Users/tiarasabrina/Documents/PROJECT/AI/address-parsing/master-data/master-matched/Master_Matching_20260701_1059 (1).xlsx", engine="openpyxl")
gold_company

,Company ID IDX,Company ID BUMN,Nama Perusahaan IDX,Nama Perusahaan BUMN,Nama Perusahaan AHU,Jaro IDX_BUMN,Jaro IDX_AHU,Jaro BUMN_AHU,Sektor,Sub Sektor,...,Kode Emiten (Ticker),Tanggal IPO (Listed),Papan Pencatatan,Alamat Lengkap,Telepon,Fax,Email,Website,NPWP,Pair Category
0,1.001,NaN,PT Adaro Andalan Indonesia Tbk,NaN,PT Adaro Andalan Indonesia,0.0,1.0,0.0,Energi,"Minyak, Gas & Batu Bara",...,AADI,2024-12-05,Utama,Cyber 2 Tower Lantai 26\nJl. H.R. Rasuna Said ...,(021) 2553 3065 | 02125533000,(021) 2553 3066,corsec@adaroindonesia.com,www.adaroindonesia.com,02.433.115.9-091.000,IDX + AHU
1,1.002,NaN,Adaro Australia Pty Ltd,NaN,NaN,0.0,0.0,0.0,Investasi,Investasi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IDX Only
2,1.003,NaN,Adaro Capital Limited,NaN,NaN,0.0,0.0,0.0,Investasi,Investasi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IDX Only
3,1.004,NaN,Adaro International (Singapore) Pte Ltd,NaN,NaN,0.0,0.0,0.0,Perdagangan batubara,Perdagangan batubara,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IDX Only
4,1.005,NaN,Arindo Holdings (Mauritius) Ltd,NaN,NaN,0.0,0.0,0.0,Investasi,Investasi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IDX Only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
772560,NaN,NaN,NaN,NaN,PT Fatima Trading Companies,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,"Gedung Fancy Mampang, Jl. Mampang Prpt. Raya N...",0881024990317,NaN,NaN,NaN,NaN,AHU Only
772561,NaN,NaN,NaN,NaN,PT Simpul Eventworks Indonesia,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,Jl. Wolter Monginsidi No.107A,087784451067,NaN,NaN,NaN,NaN,AHU Only
772562,NaN,NaN,NaN,NaN,PT Sinar Fajar Nusantara Kilat,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,Jalan Ancol Barat VIII Nomor 1 Kontener Nomor ...,081387119253,NaN,NaN,NaN,NaN,AHU Only
772563,NaN,NaN,NaN,NaN,PT Stasiun Hangat Sejahtera,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,"Infiniti Office, Bellezza BSA, 1st Floor Unit ...",082364055502,NaN,NaN,NaN,NaN,AHU Only


In [13]:
gold_company_address = gold_company[["Alamat Lengkap"]]
gold_company_address

,Alamat Lengkap
0,Cyber 2 Tower Lantai 26\nJl. H.R. Rasuna Said ...
1,NaN
2,NaN
3,NaN
4,NaN
...,...
772560,"Gedung Fancy Mampang, Jl. Mampang Prpt. Raya N..."
772561,Jl. Wolter Monginsidi No.107A
772562,Jalan Ancol Barat VIII Nomor 1 Kontener Nomor ...
772563,"Infiniti Office, Bellezza BSA, 1st Floor Unit ..."


In [14]:
#count NaN address
gold_company_address.isnull().sum()

Alamat Lengkap    6862
dtype: int64

In [19]:
df_sample = gold_company_address[
    gold_company_address["Alamat Lengkap"].str.len() > 30
].sample(n=500, random_state=42)

In [20]:
def parse_address(alamat):
    prompt = f"""
Ekstrak komponen alamat berikut ke JSON. Jika ada bagian alamat yang tidak bisa diklasifikasikan ke field di atas,
masukkan SISANYA sebagai string mentah ke field "remaining_text".
Jangan hilangkan apapun dari input.

HANYA output JSON valid, tanpa teks lain.

Alamat:
{alamat}

Format:
{{
  "gedung": "",
  "lantai": "",
  "jalan": "",
  "no": "",
  "kelurahan": "",
  "kecamatan": "",
  "kota": "",
  "provinsi": "",
  "kodepos": "",
  "remaining_text": ""
}}
"""

    try:
        r = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": "qwen2.5:7b",
                "prompt": prompt,
                "format": "json",
                "stream": False,
                "options": {
                    "temperature": 0,
                    "num_predict": 200
                }
            },
            timeout=60
        )

        return json.loads(r.json()["response"])

    except Exception as e:
        return None

In [21]:
dataset = []

for addr in tqdm(df_sample["Alamat Lengkap"]):
    parsed = parse_address(addr)

    if parsed:
        dataset.append({
            "text": addr,
            "label": parsed
        })

    time.sleep(0.05)  # biar Ollama nggak choke

100%|██████████| 500/500 [3:33:54<00:00, 25.67s/it]    


In [ ]:
import requests
import json

def parse_address(alamat):
    prompt = f"""Ekstrak alamat ke JSON.

HANYA output JSON valid.

Alamat:
{alamat}

Format:
{{"jalan": "", "no": "", "kelurahan": "", "kecamatan": "", "kota": "", "provinsi": "", "kodepos": ""}}
"""

    r = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "qwen2.5:7b",
            "prompt": prompt,
            "format": "json",
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": 200
            }
        }
    )

    try:
        return json.loads(r.json()["response"])
    except:
        return None

In [10]:
import requests
import json

def parse_address(alamat):
    prompt = f"""Ekstrak komponen alamat berikut ke JSON. HANYA JSON, tanpa penjelasan.

Alamat: "{alamat}"

Format:
{{"jalan": "", "no": "", "kelurahan": "", "kecamatan": "", "kota": "", "provinsi": "", "kodepos": ""}}"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "qwen2.5:7b",
            "prompt": prompt,
            "format": "json",
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": 150  # batasi output, ga perlu panjang
            }
        }
    )
    return json.loads(response.json()["response"])

print(parse_address("Jl. Jend. Sudirman No. 55, RT.5/RW.2, Karet Tengsin, Kecamatan Tanah Abang, Kota Jakarta Pusat, Provinsi DKI Jakarta 10220"))

{'jalan': 'Jl. Jend. Sudirman', 'no': '55', 'kelurahan': 'Karet Tengsin', 'kecamatan': 'Tanah Abang', 'kota': 'Jakarta Pusat', 'provinsi': 'DKI Jakarta', 'kodepos': '10220'}


In [9]:
!ollama ps

NAME          ID              SIZE      PROCESSOR    UNTIL                   
qwen2.5:7b    845dbda0ea48    6.3 GB    100% GPU     About a minute from now    


In [11]:
import time

for i in range(3):
    start = time.time()
    result = parse_address("Jl. Merdeka No. 10, Jakarta Pusat, DKI Jakarta 10110")
    print(f"Run {i+1}: {time.time()-start:.2f}s")

Run 1: 6.96s
Run 2: 6.12s
Run 3: 6.08s
